In [1]:
import pandas as pd
import numpy as np

def analyze_missing_data():
    """
    Analisa dados faltantes no dataset 6TARGET.csv para identificar:
    - Colunas sem dados faltantes (para Modelo 1 - amostra completa)
    - Colunas com dados faltantes (para Modelo 2 - amostra reduzida)
    """
    
    # Carregar o dataset
    dataset_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/C-Feature Engeneering/6TARGET.csv'
    df = pd.read_csv(dataset_path)
    
    print("="*80)
    print("ANÁLISE DE DADOS FALTANTES - DATASET 6TARGET.csv")
    print("="*80)
    
    # Informações gerais do dataset
    print(f"Dimensões do dataset: {df.shape}")
    print(f"Total de registros: {df.shape[0]:,}")
    print(f"Total de colunas: {df.shape[1]:,}")
    
    # Calcular dados faltantes por coluna
    missing_info = []
    
    for col in df.columns:
        missing_count = df[col].isnull().sum()
        missing_percent = (missing_count / len(df)) * 100
        data_type = str(df[col].dtype)
        unique_values = df[col].nunique()
        
        missing_info.append({
            'coluna': col,
            'total_missing': missing_count,
            'percent_missing': missing_percent,
            'tipo_dados': data_type,
            'valores_unicos': unique_values,
            'tem_missing': missing_count > 0
        })
    
    # Converter para DataFrame para análise
    missing_df = pd.DataFrame(missing_info)
    
    # Separar colunas com e sem dados faltantes
    colunas_sem_missing = missing_df[missing_df['tem_missing'] == False]['coluna'].tolist()
    colunas_com_missing = missing_df[missing_df['tem_missing'] == True]['coluna'].tolist()
    
    print(f"\n" + "="*80)
    print("RESUMO GERAL")
    print("="*80)
    print(f"Colunas SEM dados faltantes: {len(colunas_sem_missing)}")
    print(f"Colunas COM dados faltantes: {len(colunas_com_missing)}")
    
    # Identificar as variáveis target (devem estar nas colunas sem missing)
    target_vars = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']
    target_status = [var for var in target_vars if var in colunas_sem_missing]
    print(f"Variáveis target disponíveis: {len(target_status)} de {len(target_vars)}")
    
    print(f"\n" + "="*80)
    print("COLUNAS SEM DADOS FALTANTES (para Modelo 1)")
    print("="*80)
    print(f"Total: {len(colunas_sem_missing)} colunas")
    print("Listagem:")
    
    for i, col in enumerate(colunas_sem_missing, 1):
        col_info = missing_df[missing_df['coluna'] == col].iloc[0]
        print(f"{i:2d}. {col:<40} | Tipo: {col_info['tipo_dados']:<12} | Únicos: {col_info['valores_unicos']:>6}")
    
    if len(colunas_com_missing) > 0:
        print(f"\n" + "="*80)
        print("COLUNAS COM DADOS FALTANTES (para Modelo 2)")
        print("="*80)
        print(f"Total: {len(colunas_com_missing)} colunas")
        print("Listagem (ordenadas por % de missing):")
        
        # Ordenar por percentual de missing
        colunas_missing_sorted = missing_df[missing_df['tem_missing'] == True].sort_values('percent_missing')
        
        for i, (_, row) in enumerate(colunas_missing_sorted.iterrows(), 1):
            print(f"{i:2d}. {row['coluna']:<40} | Missing: {row['total_missing']:>5} ({row['percent_missing']:5.1f}%) | Tipo: {row['tipo_dados']:<12}")
    
    print(f"\n" + "="*80)
    print("ESTRATÉGIA PARA OS MODELOS")
    print("="*80)
    
    # Calcular tamanho da amostra para Modelo 2 (sem nenhum missing)
    if len(colunas_com_missing) > 0:
        # Subset sem missing em NENHUMA das colunas com missing
        df_complete_cases = df.dropna(subset=colunas_com_missing)
        modelo2_sample_size = len(df_complete_cases)
        modelo2_percent = (modelo2_sample_size / len(df)) * 100
        
        print(f"MODELO 1 (colunas sem missing):")
        print(f"  - Amostra: {len(df):,} crianças (100%)")
        print(f"  - Variáveis: {len(colunas_sem_missing)} colunas")
        print(f"  - Inclui variáveis target: {target_status}")
        
        print(f"\nMODELO 2 (todas as colunas, casos completos):")
        print(f"  - Amostra: {modelo2_sample_size:,} crianças ({modelo2_percent:.1f}%)")
        print(f"  - Variáveis: {len(colunas_sem_missing) + len(colunas_com_missing)} colunas")
        print(f"  - Redução amostral: {len(df) - modelo2_sample_size:,} crianças perdidas")
        
        # Verificar distribuição das categorias target no Modelo 2
        if all(var in df.columns for var in target_vars):
            print(f"\nDistribuição das categorias target no Modelo 2:")
            for var in target_vars:
                count_modelo2 = df_complete_cases[var].sum()
                count_original = df[var].sum()
                percent_retained = (count_modelo2 / count_original) * 100 if count_original > 0 else 0
                print(f"  - {var.capitalize()}: {count_modelo2} de {count_original} ({percent_retained:.1f}% retidos)")
    
    else:
        print(f"MODELO ÚNICO (todas as colunas sem missing):")
        print(f"  - Amostra: {len(df):,} crianças (100%)")
        print(f"  - Variáveis: {len(colunas_sem_missing)} colunas")
    
    print(f"\n" + "="*80)
    print("PRÓXIMOS PASSOS")
    print("="*80)
    print("1. Criar dataset do Modelo 1 (colunas sem missing)")
    print("2. Criar dataset do Modelo 2 (casos completos)")
    print("3. Dividir cada modelo em train/test split")
    print("4. Aplicar validação cruzada estratificada")
    
    return {
        'dataset_original': df,
        'colunas_sem_missing': colunas_sem_missing,
        'colunas_com_missing': colunas_com_missing,
        'missing_analysis': missing_df
    }

# Executar a análise
if __name__ == "__main__":
    resultado = analyze_missing_data()

ANÁLISE DE DADOS FALTANTES - DATASET 6TARGET.csv
Dimensões do dataset: (8236, 50)
Total de registros: 8,236
Total de colunas: 50

RESUMO GERAL
Colunas SEM dados faltantes: 33
Colunas COM dados faltantes: 17
Variáveis target disponíveis: 4 de 4

COLUNAS SEM DADOS FALTANTES (para Modelo 1)
Total: 33 colunas
Listagem:
 1. id_anon                                  | Tipo: int64        | Únicos:   8236
 2. regiao_Centro-Oeste                      | Tipo: int64        | Únicos:      2
 3. regiao_Nordeste                          | Tipo: int64        | Únicos:      2
 4. regiao_Norte                             | Tipo: int64        | Únicos:      2
 5. regiao_Sudeste                           | Tipo: int64        | Únicos:      2
 6. regiao_Sul                               | Tipo: int64        | Únicos:      2
 7. sexo_masculino                           | Tipo: int64        | Únicos:      2
 8. idade_crianca                            | Tipo: int64        | Únicos:      3
 9. idade_materna_a

In [2]:
import pandas as pd
import numpy as np
import os

def create_model_datasets():
    """
    Cria os datasets DSModel1.csv e DSModel2.csv:
    - DSModel1: Amostra completa (8,236) com colunas sem missing (33 variáveis)
    - DSModel2: Amostra reduzida (5,638) com colunas que tinham missing (17 + 4 target = 21 variáveis)
    """
    
    # Carregar o dataset original
    dataset_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/C-Feature Engeneering/6TARGET.csv'
    df = pd.read_csv(dataset_path)
    
    print("="*80)
    print("CRIAÇÃO DOS DATASETS MODELO 1 E MODELO 2")
    print("="*80)
    
    # Definir colunas baseado na análise anterior
    # Colunas SEM missing (para Modelo 1)
    colunas_sem_missing = [
        'id_anon', 'regiao_Centro-Oeste', 'regiao_Nordeste', 'regiao_Norte', 
        'regiao_Sudeste', 'regiao_Sul', 'sexo_masculino', 'idade_crianca',
        'idade_materna_ao_nascimento', 'cor_Branca', 'cor_Outros', 
        'cor_Parda (mulata, cabocla, cafuza, mameluca ou mestiça)', 'cor_Preta',
        'semanas_gravidez', 'peso_nascimento', 'altura_nascimento',
        'parto_Cesariana agendada (eletiva)', 'parto_Cesariana de urgência (Não agendada)',
        'parto_Normal', 'tipo_de_parto', 'chupeta_usou', 'sabe_ler',
        'sindrome_nao', 'teve_exposicao_escola', 'nivel_educacional_mae',
        'k05_prenatal_consultas', 'recebe_beneficio', 'peso_materno',
        'vd_ien_escore', 'desnutrido', 'eutrofico', 'sobrepeso', 'obeso'
    ]
    
    # Colunas COM missing (para Modelo 2) + id_anon + variáveis target
    colunas_com_missing = [
        'id_anon',  # Mantido para identificação posterior
        'k08_quilos', 'k07_peso_final', 'k06_peso_engravidar', 'k11_amamentou',
        'k03_prenatal', 'k04_prenatal_semanas', 'tempo_primeira_mamada_horas',
        'usou_utensilios_amamentacao', 'dias_aleitamento_exclusivo', 'doou_leite_banco',
        'recebeu_leite_banco', 'amamentou_outra_crianca', 'usou_mamadeira',
        'deixou_amamentar_por_outra', 'busca_info_aleitamento', 'deu_outros_liquidos',
        'k15_recebeu'
    ]
    
    # Variáveis target (necessárias em ambos os modelos)
    target_vars = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']
    
    # Verificar se todas as colunas existem no dataset
    missing_cols_modelo1 = [col for col in colunas_sem_missing if col not in df.columns]
    missing_cols_modelo2 = [col for col in colunas_com_missing if col not in df.columns]
    
    if missing_cols_modelo1:
        print(f"⚠️ Colunas não encontradas para Modelo 1: {missing_cols_modelo1}")
    if missing_cols_modelo2:
        print(f"⚠️ Colunas não encontradas para Modelo 2: {missing_cols_modelo2}")
    
    # ========================================
    # CRIAÇÃO DO DATASET MODELO 1
    # ========================================
    print(f"\n🔸 CRIANDO DATASET MODELO 1...")
    
    # Selecionar apenas colunas sem missing
    df_modelo1 = df[colunas_sem_missing].copy()
    
    print(f"Dataset Modelo 1:")
    print(f"  - Dimensões: {df_modelo1.shape}")
    print(f"  - Registros: {len(df_modelo1):,}")
    print(f"  - Colunas: {len(df_modelo1.columns)}")
    
    # Verificar se há missing no Modelo 1 (não deveria ter)
    missing_modelo1 = df_modelo1.isnull().sum().sum()
    print(f"  - Total de missing: {missing_modelo1}")
    
    # Verificar distribuição das categorias target
    print(f"  - Distribuição target:")
    for var in target_vars:
        count = df_modelo1[var].sum()
        percent = (count / len(df_modelo1)) * 100
        print(f"    {var.capitalize()}: {count} ({percent:.1f}%)")
    
    # ========================================
    # CRIAÇÃO DO DATASET MODELO 2
    # ========================================
    print(f"\n🔸 CRIANDO DATASET MODELO 2...")
    
    # Adicionar variáveis target às colunas do Modelo 2
    colunas_modelo2 = colunas_com_missing + target_vars
    
    # Selecionar colunas e remover casos com missing
    df_modelo2_temp = df[colunas_modelo2].copy()
    
    # Remover linhas com missing nas variáveis de amamentação (exceto id_anon e target)
    colunas_amamentacao = [col for col in colunas_com_missing if col != 'id_anon']
    df_modelo2 = df_modelo2_temp.dropna(subset=colunas_amamentacao)
    
    print(f"Dataset Modelo 2:")
    print(f"  - Dimensões: {df_modelo2.shape}")
    print(f"  - Registros: {len(df_modelo2):,}")
    print(f"  - Colunas: {len(df_modelo2.columns)}")
    print(f"  - Registros perdidos: {len(df) - len(df_modelo2):,}")
    print(f"  - Percentual retido: {(len(df_modelo2) / len(df)) * 100:.1f}%")
    
    # Verificar se há missing no Modelo 2 (não deveria ter após dropna)
    missing_modelo2 = df_modelo2.isnull().sum().sum()
    print(f"  - Total de missing: {missing_modelo2}")
    
    # Verificar distribuição das categorias target no Modelo 2
    print(f"  - Distribuição target:")
    for var in target_vars:
        count_modelo2 = df_modelo2[var].sum()
        count_original = df[var].sum()
        percent_modelo2 = (count_modelo2 / len(df_modelo2)) * 100
        percent_retido = (count_modelo2 / count_original) * 100 if count_original > 0 else 0
        print(f"    {var.capitalize()}: {count_modelo2} ({percent_modelo2:.1f}%) - {percent_retido:.1f}% do original")
    
    # ========================================
    # SALVAR OS DATASETS
    # ========================================
    print(f"\n🔸 SALVANDO DATASETS...")
    
    # Criar diretório se não existir
    output_dir = '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split'
    os.makedirs(output_dir, exist_ok=True)
    
    # Salvar Modelo 1
    modelo1_path = os.path.join(output_dir, 'DSModel1.csv')
    df_modelo1.to_csv(modelo1_path, index=False)
    print(f"✅ DSModel1.csv salvo: {modelo1_path}")
    print(f"   Dimensões: {df_modelo1.shape}")
    
    # Salvar Modelo 2
    modelo2_path = os.path.join(output_dir, 'DSModel2.csv')
    df_modelo2.to_csv(modelo2_path, index=False)
    print(f"✅ DSModel2.csv salvo: {modelo2_path}")
    print(f"   Dimensões: {df_modelo2.shape}")
    
    print(f"\n" + "="*80)
    print("RESUMO FINAL")
    print("="*80)
    print(f"DSModel1.csv: {df_modelo1.shape[0]:,} registros × {df_modelo1.shape[1]} colunas")
    print(f"  - Variáveis demográficas, perinatais e socioeconômicas")
    print(f"  - Amostra completa (100%)")
    print(f"  - Sem dados faltantes")
    
    print(f"\nDSModel2.csv: {df_modelo2.shape[0]:,} registros × {df_modelo2.shape[1]} colunas")
    print(f"  - Variáveis de amamentação e práticas alimentares + id_anon")
    print(f"  - Amostra reduzida ({(len(df_modelo2) / len(df)) * 100:.1f}%)")
    print(f"  - Casos completos (sem missing)")
    print(f"  - Inclui id_anon para identificação posterior")
    
    print(f"\n🎯 Próximo passo: Train-Test Split para ambos os datasets")
    
    return df_modelo1, df_modelo2

# Executar a criação dos datasets
if __name__ == "__main__":
    modelo1, modelo2 = create_model_datasets()

CRIAÇÃO DOS DATASETS MODELO 1 E MODELO 2

🔸 CRIANDO DATASET MODELO 1...
Dataset Modelo 1:
  - Dimensões: (8236, 33)
  - Registros: 8,236
  - Colunas: 33
  - Total de missing: 0
  - Distribuição target:
    Desnutrido: 222 (2.7%)
    Eutrofico: 7327 (89.0%)
    Sobrepeso: 458 (5.6%)
    Obeso: 229 (2.8%)

🔸 CRIANDO DATASET MODELO 2...
Dataset Modelo 2:
  - Dimensões: (5638, 22)
  - Registros: 5,638
  - Colunas: 22
  - Registros perdidos: 2,598
  - Percentual retido: 68.5%
  - Total de missing: 0
  - Distribuição target:
    Desnutrido: 157 (2.8%) - 70.7% do original
    Eutrofico: 4995 (88.6%) - 68.2% do original
    Sobrepeso: 318 (5.6%) - 69.4% do original
    Obeso: 168 (3.0%) - 73.4% do original

🔸 SALVANDO DATASETS...
✅ DSModel1.csv salvo: /Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/DSModel1.csv
   Dimensões: (8236, 33)
✅ DSModel2.csv salvo: /Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/DSModel2.csv
   Dimensões: (5638, 22)

RES

# Criação dos Datasets para Modelos de Predição de Obesidade Infantil

## 📋 Contexto do Estudo

Este documento descreve a criação de dois datasets distintos para análise de fatores associados à obesidade e desnutrição infantil, baseado nos dados do **ENANI-2019** (Estudo Nacional de Alimentação e Nutrição Infantil).

### Objetivo
Desenvolver dois modelos complementares para identificar fatores associados ao estado nutricional de crianças de 2-4 anos:
- **Modelo 1**: Fatores demográficos, perinatais e socioeconômicos
- **Modelo 2**: Práticas de amamentação e alimentação complementar

---

## 🔍 Análise Inicial dos Dados

### Dataset Original: `6TARGET.csv`
- **Dimensões**: 8.236 registros × 50 colunas
- **População**: Crianças brasileiras de 2-4 anos (ENANI-2019)
- **Variáveis Target**: 4 categorias nutricionais baseadas no BMI-for-age z-score (WHO)

### Distribuição das Categorias Target
| Categoria | N | % |
|-----------|---|---|
| **Eutrófico** | 7.327 | 89.0% |
| **Sobrepeso** | 458 | 5.6% |
| **Obeso** | 229 | 2.8% |
| **Desnutrido** | 222 | 2.7% |

### Padrão de Dados Faltantes
- **33 colunas** sem dados faltantes (0%)
- **17 colunas** com dados faltantes (25-30%)
- **Padrão sistemático**: Missing concentrado em variáveis de amamentação

---

## 📊 Estratégia de Divisão dos Datasets

### Rationale Metodológico
1. **Maximizar poder estatístico** para análise populacional geral
2. **Preservar especificidade** para análise de práticas alimentares
3. **Evitar imputação** que poderia introduzir viés
4. **Manter representatividade** das categorias nutricionais

### Critérios de Seleção
- **Modelo 1**: Variáveis com 100% de completude
- **Modelo 2**: Casos com dados completos para variáveis de amamentação

---

## 🗂️ Datasets Criados

### DSModel1.csv - Modelo Demográfico/Perinatal
```
📁 Localização: /Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/
📏 Dimensões: 8.236 registros × 33 colunas
🎯 Amostra: 100% (completa)
💾 Dados faltantes: 0%
```

#### Categorias de Variáveis Incluídas:
- **Demográficas**: região, sexo, idade da criança
- **Maternas**: idade materna ao nascimento, cor/raça, educação
- **Perinatais**: semanas de gravidez, peso/altura ao nascer, tipo de parto
- **Socioeconômicas**: nível educacional, benefícios governamentais
- **Antropométricas**: peso materno, escore socioeconômico
- **Outras**: uso de chupeta, síndromes, consultas pré-natal
- **Target**: 4 variáveis binárias (desnutrido, eutrófico, sobrepeso, obeso)

### DSModel2.csv - Modelo de Práticas Alimentares
```
📁 Localização: /Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/
📏 Dimensões: 5.638 registros × 22 colunas
🎯 Amostra: 68.5% (casos completos)
💾 Dados faltantes: 0% (após exclusão)
```

#### Categorias de Variáveis Incluídas:
- **Identificação**: id_anon (para linkage posterior)
- **Peso materno**: ganho de peso gestacional
- **Cuidado pré-natal**: consultas e acompanhamento
- **Amamentação**: duração exclusiva, primeira mamada, utensílios
- **Banco de leite**: doação e recebimento
- **Práticas alimentares**: outros líquidos, mamadeira
- **Comportamento**: busca de informações sobre aleitamento
- **Target**: 4 variáveis binárias (mesmas do Modelo 1)

---

## 📈 Impacto da Redução Amostral (Modelo 2)

### Retenção por Categoria Nutricional
| Categoria | Original | Modelo 2 | % Retido | Status |
|-----------|----------|----------|----------|--------|
| **Desnutrido** | 222 | 157 | 70.7% | ✅ Bom |
| **Eutrófico** | 7.327 | 4.995 | 68.2% | ✅ Adequado |
| **Sobrepeso** | 458 | 318 | 69.4% | ✅ Bom |
| **Obeso** | 229 | 168 | 73.4% | ✅ Muito bom |

### Características da Redução:
- **Perda balanceada**: Todas as categorias mantêm ~70% dos casos
- **Sem viés sistemático**: Distribuição proporcional preservada
- **Poder estatístico mantido**: Ainda adequado para análises

---

## 🎯 Vantagens da Estratégia Adotada

### Modelo 1 (Amostra Completa)
✅ **Máximo poder estatístico** (n=8.236)  
✅ **Representatividade nacional** preservada  
✅ **Análise populacional** robusta  
✅ **Comparabilidade internacional** (variáveis padrão)  

### Modelo 2 (Práticas Alimentares)
✅ **Especificidade temática** (foco em amamentação)  
✅ **Dados de alta qualidade** (casos completos)  
✅ **Relevância clínica** (práticas modificáveis)  
✅ **Identificação preservada** (id_anon para linkage)  

---

## 🔄 Próximos Passos

### Preparação para Machine Learning
1. **Train-Test Split** estratificado para ambos os datasets
2. **Validação cruzada** com preservação da distribuição das classes
3. **Feature engineering** específico para cada modelo
4. **Estratégias de balanceamento** para classes minoritárias

### Análises Planejadas
- **Modelo 1**: Identificação de fatores de risco não modificáveis
- **Modelo 2**: Avaliação de práticas alimentares modificáveis
- **Análise comparativa**: Poder preditivo entre os modelos
- **Interpretabilidade**: SHAP values e importância de features

---

## 📝 Considerações Metodológicas

### Pontos Fortes
- Estratégia baseada em padrões reais de missing data
- Preservação da representatividade amostral
- Abordagem transparente sem imputação artificial
- Foco em questões clinicamente relevantes

### Limitações
- Redução amostral no Modelo 2 pode limitar algumas análises
- Possível viés de seleção em famílias que responderam questões de amamentação
- Classes desbalanceadas requerem técnicas específicas de ML

### Validação
- Comparação das características entre amostras completa e reduzida
- Análise de sensibilidade para missing data patterns
- Validação externa com outros estudos brasileiros

---

## 📚 Referências Metodológicas

- **WHO Child Growth Standards** para classificação nutricional
- **ENANI-2019** protocolo e amostragem
- **Diretrizes STROBE** para estudos observacionais
- **Boas práticas** em machine learning para saúde pública

---

*Documento criado em: Setembro 2025*  
*Projeto: Early Obesity Prediction - Brazilian Children*  
*Base: ENANI-2019 Secondary Analysis*

In [3]:
import pandas as pd

def remove_peso_materno_from_dsmodel1():
    """
    Remove a variável peso_materno do DSModel1.csv por ser do momento da pesquisa
    (não dos primeiros 24 meses de vida) e salva sobre o próprio arquivo
    """
    
    # Carregar o DSModel1
    dataset_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/DSModel1.csv'
    df = pd.read_csv(dataset_path)
    
    print("="*70)
    print("REMOÇÃO DA VARIÁVEL peso_materno - DSModel1")
    print("="*70)
    
    # Informações antes da remoção
    print(f"Dimensões ANTES: {df.shape}")
    print(f"Colunas ANTES: {df.shape[1]}")
    
    # Verificar se a coluna existe
    if 'peso_materno' in df.columns:
        print(f"✓ Coluna 'peso_materno' encontrada")
        
        # Mostrar algumas estatísticas da variável que será removida
        print(f"\nEstatísticas da variável peso_materno (que será removida):")
        print(f"  - Valores não nulos: {df['peso_materno'].notna().sum():,}")
        print(f"  - Valores nulos: {df['peso_materno'].isna().sum():,}")
        print(f"  - Média: {df['peso_materno'].mean():.1f} kg")
        print(f"  - Min: {df['peso_materno'].min():.1f} kg")
        print(f"  - Max: {df['peso_materno'].max():.1f} kg")
        
        # Remover a coluna
        df_cleaned = df.drop('peso_materno', axis=1)
        
        print(f"\n🗑️ Coluna 'peso_materno' removida com sucesso!")
        
    else:
        print(f"⚠️ Coluna 'peso_materno' não encontrada no dataset")
        df_cleaned = df.copy()
    
    # Informações após a remoção
    print(f"\nDimensões DEPOIS: {df_cleaned.shape}")
    print(f"Colunas DEPOIS: {df_cleaned.shape[1]}")
    print(f"Redução: {df.shape[1] - df_cleaned.shape[1]} coluna removida")
    
    # Verificar integridade das variáveis target
    target_vars = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']
    print(f"\n✓ Verificação das variáveis target:")
    for var in target_vars:
        if var in df_cleaned.columns:
            count = df_cleaned[var].sum()
            print(f"  - {var.capitalize()}: {count} casos")
        else:
            print(f"  - ⚠️ {var} não encontrada!")
    
    # Listar as colunas finais
    print(f"\nColunas finais no DSModel1 ({len(df_cleaned.columns)} total):")
    for i, col in enumerate(df_cleaned.columns, 1):
        print(f"{i:2d}. {col}")
    
    # Salvar sobre o próprio arquivo
    df_cleaned.to_csv(dataset_path, index=False)
    print(f"\n✅ Arquivo atualizado e salvo: {dataset_path}")
    print(f"   Dimensões finais: {df_cleaned.shape}")
    
    print(f"\n" + "="*70)
    print("JUSTIFICATIVA DA REMOÇÃO")
    print("="*70)
    print("A variável 'peso_materno' foi removida porque:")
    print("• Refere-se ao peso da mãe no MOMENTO DA PESQUISA")
    print("• Criança já tem 2-4 anos de idade")
    print("• NÃO se relaciona com os primeiros 24 meses de vida")
    print("• Pode causar data leakage no modelo")
    print("• Foco deve ser em fatores dos primeiros 24 meses")
    
    return df_cleaned

# Executar a remoção
if __name__ == "__main__":
    df_final = remove_peso_materno_from_dsmodel1()

REMOÇÃO DA VARIÁVEL peso_materno - DSModel1
Dimensões ANTES: (8236, 33)
Colunas ANTES: 33
✓ Coluna 'peso_materno' encontrada

Estatísticas da variável peso_materno (que será removida):
  - Valores não nulos: 8,236
  - Valores nulos: 0
  - Média: 69.5 kg
  - Min: 31.5 kg
  - Max: 178.0 kg

🗑️ Coluna 'peso_materno' removida com sucesso!

Dimensões DEPOIS: (8236, 32)
Colunas DEPOIS: 32
Redução: 1 coluna removida

✓ Verificação das variáveis target:
  - Desnutrido: 222 casos
  - Eutrofico: 7327 casos
  - Sobrepeso: 458 casos
  - Obeso: 229 casos

Colunas finais no DSModel1 (32 total):
 1. id_anon
 2. regiao_Centro-Oeste
 3. regiao_Nordeste
 4. regiao_Norte
 5. regiao_Sudeste
 6. regiao_Sul
 7. sexo_masculino
 8. idade_crianca
 9. idade_materna_ao_nascimento
10. cor_Branca
11. cor_Outros
12. cor_Parda (mulata, cabocla, cafuza, mameluca ou mestiça)
13. cor_Preta
14. semanas_gravidez
15. peso_nascimento
16. altura_nascimento
17. parto_Cesariana agendada (eletiva)
18. parto_Cesariana de urgênc

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

def create_stratified_train_test_model1():
    """
    Implementa train-test split estratificado para DSModel1 mantendo 
    distribuição idêntica das 4 classes nutricionais em ambos os conjuntos
    """
    
    # Carregar o DSModel1
    dataset_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/DSModel1.csv'
    df = pd.read_csv(dataset_path)
    
    print("="*80)
    print("TRAIN-TEST SPLIT ESTRATIFICADO - DSMODEL1")
    print("="*80)
    
    # Informações do dataset original
    print(f"Dataset original:")
    print(f"  - Dimensões: {df.shape}")
    print(f"  - Total de registros: {len(df):,}")
    
    # Definir variáveis target
    target_vars = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']
    
    # Verificar se todas as variáveis target existem
    missing_targets = [var for var in target_vars if var not in df.columns]
    if missing_targets:
        print(f"❌ ERRO: Variáveis target não encontradas: {missing_targets}")
        return None, None
    
    # Criar variável target única para stratificação
    # Converter matriz binária em categoria única (0=desnutrido, 1=eutrofico, 2=sobrepeso, 3=obeso)
    target_combined = (df['desnutrido'] * 0 + 
                      df['eutrofico'] * 1 + 
                      df['sobrepeso'] * 2 + 
                      df['obeso'] * 3)
    
    # Verificar distribuição original das classes
    print(f"\nDistribuição original das classes:")
    for i, var in enumerate(target_vars):
        count = df[var].sum()
        percent = (count / len(df)) * 100
        print(f"  - {var.capitalize()}: {count:,} ({percent:.1f}%)")
    
    print(f"  - Total: {target_combined.value_counts().sum():,}")
    
    # Preparar features (todas exceto target variables)
    feature_cols = [col for col in df.columns if col not in target_vars]
    X = df[feature_cols].copy()
    y = target_combined.copy()
    ids = df['id_anon'].copy()
    
    print(f"\nFeatures selecionadas: {len(feature_cols)} colunas")
    print(f"  - Inclui id_anon: {'id_anon' in feature_cols}")
    
    # Implementar train-test split estratificado
    print(f"\n🎲 Executando train-test split estratificado...")
    print(f"  - Proporção: 80% train / 20% test")
    print(f"  - Random state: 42")
    print(f"  - Stratify: pelas 4 classes nutricionais")
    
    X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
        X, y, ids,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    
    # Reconverter y_train e y_test de volta para matriz binária
    def convert_to_binary_matrix(y_series, target_vars):
        """Converte target único de volta para matriz binária"""
        n_samples = len(y_series)
        matrix = pd.DataFrame(0, index=y_series.index, columns=target_vars)
        
        for i, var in enumerate(target_vars):
            matrix.loc[y_series == i, var] = 1
        
        return matrix
    
    y_train_matrix = convert_to_binary_matrix(y_train, target_vars)
    y_test_matrix = convert_to_binary_matrix(y_test, target_vars)
    
    # Criar datasets finais combinando features + target
    train_final = pd.concat([X_train, y_train_matrix], axis=1)
    test_final = pd.concat([X_test, y_test_matrix], axis=1)
    
    print(f"\n📊 RESULTADOS DO SPLIT:")
    print(f"Training set: {len(train_final):,} observações ({len(train_final)/len(df)*100:.1f}%)")
    print(f"Test set: {len(test_final):,} observações ({len(test_final)/len(df)*100:.1f}%)")
    
    # Verificar distribuição das classes em cada conjunto
    print(f"\nDistribuição no TRAINING SET:")
    for var in target_vars:
        count = train_final[var].sum()
        percent = (count / len(train_final)) * 100
        print(f"  - {var.capitalize()}: {count:,} ({percent:.1f}%)")
    
    print(f"\nDistribuição no TEST SET:")
    for var in target_vars:
        count = test_final[var].sum()
        percent = (count / len(test_final)) * 100
        print(f"  - {var.capitalize()}: {count:,} ({percent:.1f}%)")
    
    # Quality Assurance
    print(f"\n🔍 QUALITY ASSURANCE:")
    
    # Verificar sobreposição de IDs
    id_overlap = set(ids_train) & set(ids_test)
    print(f"  - ID overlap entre train/test: {len(id_overlap)} (deve ser 0)")
    
    # Verificar preservação total da amostra
    total_samples = len(train_final) + len(test_final)
    print(f"  - Total de amostras preservadas: {total_samples:,} de {len(df):,}")
    print(f"  - Preservação: {'✅ 100%' if total_samples == len(df) else '❌ Erro'}")
    
    # Verificar se cada linha tem exatamente 1 categoria ativa
    train_sums = train_final[target_vars].sum(axis=1)
    test_sums = test_final[target_vars].sum(axis=1)
    
    print(f"  - Linhas train com exatamente 1 categoria: {(train_sums == 1).sum()} de {len(train_final)}")
    print(f"  - Linhas test com exatamente 1 categoria: {(test_sums == 1).sum()} de {len(test_final)}")
    
    # Comparar proporções originais vs. splits
    print(f"\n📈 COMPARAÇÃO DE PROPORÇÕES:")
    print(f"{'Categoria':<12} {'Original':<10} {'Train':<10} {'Test':<10} {'Status':<8}")
    print("-" * 55)
    
    for var in target_vars:
        orig_pct = (df[var].sum() / len(df)) * 100
        train_pct = (train_final[var].sum() / len(train_final)) * 100
        test_pct = (test_final[var].sum() / len(test_final)) * 100
        
        # Verificar se as proporções são similares (±0.5%)
        status = "✅" if abs(orig_pct - train_pct) < 0.5 and abs(orig_pct - test_pct) < 0.5 else "⚠️"
        
        print(f"{var.capitalize():<12} {orig_pct:<10.1f} {train_pct:<10.1f} {test_pct:<10.1f} {status:<8}")
    
    # Salvar os datasets
    output_dir = '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split'
    
    train_path = os.path.join(output_dir, 'MODEL1TRAIN.csv')
    test_path = os.path.join(output_dir, 'MODEL1TEST.csv')
    
    train_final.to_csv(train_path, index=False)
    test_final.to_csv(test_path, index=False)
    
    print(f"\n✅ DATASETS SALVOS:")
    print(f"  - Training: {train_path}")
    print(f"    Dimensões: {train_final.shape}")
    print(f"  - Test: {test_path}")
    print(f"    Dimensões: {test_final.shape}")
    
    print(f"\n" + "="*80)
    print("METODOLOGIA PARA PAPER")
    print("="*80)
    methodology_text = f"""
Prior to machine learning analysis, we implemented a stratified train-test split using 
sklearn.train_test_split with stratify=y parameter to maintain identical nutritional 
status distribution across both sets. We set random_state=42 for reproducible results 
and preserved child IDs throughout the split to verify no data leakage through triple 
splitting of features, target, and IDs simultaneously. This methodology yielded a 
training set of {len(train_final):,} observations (80.0%) with {train_final['desnutrido'].sum()} 
undernourished ({(train_final['desnutrido'].sum()/len(train_final)*100):.1f}%), 
{train_final['eutrofico'].sum():,} normal weight ({(train_final['eutrofico'].sum()/len(train_final)*100):.1f}%), 
{train_final['sobrepeso'].sum()} overweight ({(train_final['sobrepeso'].sum()/len(train_final)*100):.1f}%), 
and {train_final['obeso'].sum()} obese ({(train_final['obeso'].sum()/len(train_final)*100):.1f}%), 
and a hold-out test set of {len(test_final):,} observations (20.0%) with {test_final['desnutrido'].sum()} 
undernourished ({(test_final['desnutrido'].sum()/len(test_final)*100):.1f}%), 
{test_final['eutrofico'].sum():,} normal weight ({(test_final['eutrofico'].sum()/len(test_final)*100):.1f}%), 
{test_final['sobrepeso'].sum()} overweight ({(test_final['sobrepeso'].sum()/len(test_final)*100):.1f}%), 
and {test_final['obeso'].sum()} obese ({(test_final['obeso'].sum()/len(test_final)*100):.1f}%). 
Quality assurance verified zero ID overlap between sets, confirmed total sample preservation, 
and validated identical target proportions across all four nutritional categories. The 
hold-out test set remained completely isolated from all subsequent preprocessing, feature 
selection, and model training procedures.
"""
    print(methodology_text)
    
    return train_final, test_final

# Executar o train-test split
if __name__ == "__main__":
    train_data, test_data = create_stratified_train_test_model1()

TRAIN-TEST SPLIT ESTRATIFICADO - DSMODEL1
Dataset original:
  - Dimensões: (8236, 32)
  - Total de registros: 8,236

Distribuição original das classes:
  - Desnutrido: 222 (2.7%)
  - Eutrofico: 7,327 (89.0%)
  - Sobrepeso: 458 (5.6%)
  - Obeso: 229 (2.8%)
  - Total: 8,236

Features selecionadas: 28 colunas
  - Inclui id_anon: True

🎲 Executando train-test split estratificado...
  - Proporção: 80% train / 20% test
  - Random state: 42
  - Stratify: pelas 4 classes nutricionais

📊 RESULTADOS DO SPLIT:
Training set: 6,588 observações (80.0%)
Test set: 1,648 observações (20.0%)

Distribuição no TRAINING SET:
  - Desnutrido: 178 (2.7%)
  - Eutrofico: 5,861 (89.0%)
  - Sobrepeso: 366 (5.6%)
  - Obeso: 183 (2.8%)

Distribuição no TEST SET:
  - Desnutrido: 44 (2.7%)
  - Eutrofico: 1,466 (89.0%)
  - Sobrepeso: 92 (5.6%)
  - Obeso: 46 (2.8%)

🔍 QUALITY ASSURANCE:
  - ID overlap entre train/test: 0 (deve ser 0)
  - Total de amostras preservadas: 8,236 de 8,236
  - Preservação: ✅ 100%
  - Linhas t

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

def create_stratified_train_test_model2():
    """
    Implementa train-test split estratificado para DSModel2 mantendo 
    distribuição idêntica das 4 classes nutricionais em ambos os conjuntos
    """
    
    # Carregar o DSModel2
    dataset_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/DSModel2.csv'
    df = pd.read_csv(dataset_path)
    
    print("="*80)
    print("TRAIN-TEST SPLIT ESTRATIFICADO - DSMODEL2")
    print("="*80)
    
    # Informações do dataset original
    print(f"Dataset original:")
    print(f"  - Dimensões: {df.shape}")
    print(f"  - Total de registros: {len(df):,}")
    
    # Definir variáveis target
    target_vars = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']
    
    # Verificar se todas as variáveis target existem
    missing_targets = [var for var in target_vars if var not in df.columns]
    if missing_targets:
        print(f"❌ ERRO: Variáveis target não encontradas: {missing_targets}")
        return None, None
    
    # Criar variável target única para stratificação
    # Converter matriz binária em categoria única (0=desnutrido, 1=eutrofico, 2=sobrepeso, 3=obeso)
    target_combined = (df['desnutrido'] * 0 + 
                      df['eutrofico'] * 1 + 
                      df['sobrepeso'] * 2 + 
                      df['obeso'] * 3)
    
    # Verificar distribuição original das classes
    print(f"\nDistribuição original das classes:")
    for i, var in enumerate(target_vars):
        count = df[var].sum()
        percent = (count / len(df)) * 100
        print(f"  - {var.capitalize()}: {count:,} ({percent:.1f}%)")
    
    print(f"  - Total: {target_combined.value_counts().sum():,}")
    
    # Preparar features (todas exceto target variables)
    feature_cols = [col for col in df.columns if col not in target_vars]
    X = df[feature_cols].copy()
    y = target_combined.copy()
    ids = df['id_anon'].copy()
    
    print(f"\nFeatures selecionadas: {len(feature_cols)} colunas")
    print(f"  - Inclui id_anon: {'id_anon' in feature_cols}")
    print(f"  - Variáveis de amamentação: {len(feature_cols) - 1}")  # -1 por causa do id_anon
    
    # Implementar train-test split estratificado
    print(f"\n🎲 Executando train-test split estratificado...")
    print(f"  - Proporção: 80% train / 20% test")
    print(f"  - Random state: 42")
    print(f"  - Stratify: pelas 4 classes nutricionais")
    
    X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
        X, y, ids,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    
    # Reconverter y_train e y_test de volta para matriz binária
    def convert_to_binary_matrix(y_series, target_vars):
        """Converte target único de volta para matriz binária"""
        n_samples = len(y_series)
        matrix = pd.DataFrame(0, index=y_series.index, columns=target_vars)
        
        for i, var in enumerate(target_vars):
            matrix.loc[y_series == i, var] = 1
        
        return matrix
    
    y_train_matrix = convert_to_binary_matrix(y_train, target_vars)
    y_test_matrix = convert_to_binary_matrix(y_test, target_vars)
    
    # Criar datasets finais combinando features + target
    train_final = pd.concat([X_train, y_train_matrix], axis=1)
    test_final = pd.concat([X_test, y_test_matrix], axis=1)
    
    print(f"\n📊 RESULTADOS DO SPLIT:")
    print(f"Training set: {len(train_final):,} observações ({len(train_final)/len(df)*100:.1f}%)")
    print(f"Test set: {len(test_final):,} observações ({len(test_final)/len(df)*100:.1f}%)")
    
    # Verificar distribuição das classes em cada conjunto
    print(f"\nDistribuição no TRAINING SET:")
    for var in target_vars:
        count = train_final[var].sum()
        percent = (count / len(train_final)) * 100
        print(f"  - {var.capitalize()}: {count:,} ({percent:.1f}%)")
    
    print(f"\nDistribuição no TEST SET:")
    for var in target_vars:
        count = test_final[var].sum()
        percent = (count / len(test_final)) * 100
        print(f"  - {var.capitalize()}: {count:,} ({percent:.1f}%)")
    
    # Quality Assurance
    print(f"\n🔍 QUALITY ASSURANCE:")
    
    # Verificar sobreposição de IDs
    id_overlap = set(ids_train) & set(ids_test)
    print(f"  - ID overlap entre train/test: {len(id_overlap)} (deve ser 0)")
    
    # Verificar preservação total da amostra
    total_samples = len(train_final) + len(test_final)
    print(f"  - Total de amostras preservadas: {total_samples:,} de {len(df):,}")
    print(f"  - Preservação: {'✅ 100%' if total_samples == len(df) else '❌ Erro'}")
    
    # Verificar se cada linha tem exatamente 1 categoria ativa
    train_sums = train_final[target_vars].sum(axis=1)
    test_sums = test_final[target_vars].sum(axis=1)
    
    print(f"  - Linhas train com exatamente 1 categoria: {(train_sums == 1).sum()} de {len(train_final)}")
    print(f"  - Linhas test com exatamente 1 categoria: {(test_sums == 1).sum()} de {len(test_final)}")
    
    # Comparar proporções originais vs. splits
    print(f"\n📈 COMPARAÇÃO DE PROPORÇÕES:")
    print(f"{'Categoria':<12} {'Original':<10} {'Train':<10} {'Test':<10} {'Status':<8}")
    print("-" * 55)
    
    for var in target_vars:
        orig_pct = (df[var].sum() / len(df)) * 100
        train_pct = (train_final[var].sum() / len(train_final)) * 100
        test_pct = (test_final[var].sum() / len(test_final)) * 100
        
        # Verificar se as proporções são similares (±0.5%)
        status = "✅" if abs(orig_pct - train_pct) < 0.5 and abs(orig_pct - test_pct) < 0.5 else "⚠️"
        
        print(f"{var.capitalize():<12} {orig_pct:<10.1f} {train_pct:<10.1f} {test_pct:<10.1f} {status:<8}")
    
    # Comparação com Modelo 1 (para contexto)
    print(f"\n📋 COMPARAÇÃO COM MODELO 1:")
    print(f"  - Modelo 1 (amostra completa): 8,236 crianças")
    print(f"  - Modelo 2 (casos completos): {len(df):,} crianças")
    print(f"  - Redução amostral: {8236 - len(df):,} crianças ({((8236 - len(df))/8236)*100:.1f}%)")
    print(f"  - Retenção: {(len(df)/8236)*100:.1f}%")
    
    # Salvar os datasets
    output_dir = '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split'
    
    train_path = os.path.join(output_dir, 'MODEL2TRAIN.csv')
    test_path = os.path.join(output_dir, 'MODEL2TEST.csv')
    
    train_final.to_csv(train_path, index=False)
    test_final.to_csv(test_path, index=False)
    
    print(f"\n✅ DATASETS SALVOS:")
    print(f"  - Training: {train_path}")
    print(f"    Dimensões: {train_final.shape}")
    print(f"  - Test: {test_path}")
    print(f"    Dimensões: {test_final.shape}")
    
    print(f"\n" + "="*80)
    print("METODOLOGIA PARA PAPER - MODELO 2")
    print("="*80)
    methodology_text = f"""
For Model 2 (feeding practices analysis), we applied the same stratified train-test split 
methodology to the subset of {len(df):,} children with complete breastfeeding and feeding 
practice data. Using sklearn.train_test_split with stratify=y parameter and random_state=42, 
we maintained identical nutritional status distribution across both sets while preserving 
child IDs to verify no data leakage. This approach yielded a training set of {len(train_final):,} 
observations (80.0%) with {train_final['desnutrido'].sum()} undernourished 
({(train_final['desnutrido'].sum()/len(train_final)*100):.1f}%), {train_final['eutrofico'].sum():,} 
normal weight ({(train_final['eutrofico'].sum()/len(train_final)*100):.1f}%), 
{train_final['sobrepeso'].sum()} overweight ({(train_final['sobrepeso'].sum()/len(train_final)*100):.1f}%), 
and {train_final['obeso'].sum()} obese ({(train_final['obeso'].sum()/len(train_final)*100):.1f}%), 
and a hold-out test set of {len(test_final):,} observations (20.0%) with {test_final['desnutrido'].sum()} 
undernourished ({(test_final['desnutrido'].sum()/len(test_final)*100):.1f}%), 
{test_final['eutrofico'].sum():,} normal weight ({(test_final['eutrofico'].sum()/len(test_final)*100):.1f}%), 
{test_final['sobrepeso'].sum()} overweight ({(test_final['sobrepeso'].sum()/len(test_final)*100):.1f}%), 
and {test_final['obeso'].sum()} obese ({(test_final['obeso'].sum()/len(test_final)*100):.1f}%). 
Quality assurance confirmed zero ID overlap, total sample preservation, and identical target 
proportions across all nutritional categories. Both Model 2 datasets remained isolated from 
preprocessing and model training procedures until final evaluation.
"""
    print(methodology_text)
    
    print(f"\n" + "="*80)
    print("RESUMO COMPARATIVO DOS MODELOS")
    print("="*80)
    print(f"MODELO 1 (Fatores Demográficos/Perinatais):")
    print(f"  - Amostra: 8,236 crianças (100% da população ENANI-2019)")
    print(f"  - Training: 6,588 | Test: 1,648")
    print(f"  - Variáveis: 28 features + 4 target")
    print(f"  - Foco: Fatores não modificáveis dos primeiros 24 meses")
    
    print(f"\nMODELO 2 (Práticas de Alimentação):")
    print(f"  - Amostra: {len(df):,} crianças ({(len(df)/8236)*100:.1f}% da população)")
    print(f"  - Training: {len(train_final):,} | Test: {len(test_final):,}")
    print(f"  - Variáveis: {len(feature_cols)} features + 4 target")
    print(f"  - Foco: Práticas de amamentação e alimentação modificáveis")
    
    return train_final, test_final

# Executar o train-test split para Modelo 2
if __name__ == "__main__":
    train_data_m2, test_data_m2 = create_stratified_train_test_model2()

TRAIN-TEST SPLIT ESTRATIFICADO - DSMODEL2
Dataset original:
  - Dimensões: (5638, 22)
  - Total de registros: 5,638

Distribuição original das classes:
  - Desnutrido: 157 (2.8%)
  - Eutrofico: 4,995 (88.6%)
  - Sobrepeso: 318 (5.6%)
  - Obeso: 168 (3.0%)
  - Total: 5,638

Features selecionadas: 18 colunas
  - Inclui id_anon: True
  - Variáveis de amamentação: 17

🎲 Executando train-test split estratificado...
  - Proporção: 80% train / 20% test
  - Random state: 42
  - Stratify: pelas 4 classes nutricionais

📊 RESULTADOS DO SPLIT:
Training set: 4,510 observações (80.0%)
Test set: 1,128 observações (20.0%)

Distribuição no TRAINING SET:
  - Desnutrido: 126 (2.8%)
  - Eutrofico: 3,996 (88.6%)
  - Sobrepeso: 254 (5.6%)
  - Obeso: 134 (3.0%)

Distribuição no TEST SET:
  - Desnutrido: 31 (2.7%)
  - Eutrofico: 999 (88.6%)
  - Sobrepeso: 64 (5.7%)
  - Obeso: 34 (3.0%)

🔍 QUALITY ASSURANCE:
  - ID overlap entre train/test: 0 (deve ser 0)
  - Total de amostras preservadas: 5,638 de 5,638
  - P

In [6]:
import pandas as pd
import numpy as np
import os

def create_model3_combined_dataset():
    """
    Cria o Modelo 3 com todas as variáveis, mantendo apenas casos completos
    (sem nenhum valor missing em qualquer coluna)
    """
    
    # Carregar o dataset original completo
    original_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/C-Feature Engeneering/6TARGET.csv'
    df = pd.read_csv(original_path)
    
    print("="*80)
    print("CRIAÇÃO DO MODELO 3 - DATASET COMBINADO")
    print("="*80)
    
    # Informações do dataset original
    print(f"Dataset original:")
    print(f"  - Dimensões: {df.shape}")
    print(f"  - Total de colunas: {df.shape[1]}")
    print(f"  - Total de registros: {df.shape[0]:,}")
    
    # Analisar missing values por coluna
    missing_info = []
    for col in df.columns:
        missing_count = df[col].isnull().sum()
        missing_percent = (missing_count / len(df)) * 100
        missing_info.append({
            'coluna': col,
            'missing_count': missing_count,
            'missing_percent': missing_percent
        })
    
    missing_df = pd.DataFrame(missing_info)
    colunas_com_missing = missing_df[missing_df['missing_count'] > 0]['coluna'].tolist()
    colunas_sem_missing = missing_df[missing_df['missing_count'] == 0]['coluna'].tolist()
    
    print(f"\nAnálise de missing values:")
    print(f"  - Colunas SEM missing: {len(colunas_sem_missing)}")
    print(f"  - Colunas COM missing: {len(colunas_com_missing)}")
    
    if colunas_com_missing:
        print(f"\nColunas com missing values:")
        for col in colunas_com_missing:
            col_info = missing_df[missing_df['coluna'] == col].iloc[0]
            print(f"  - {col}: {col_info['missing_count']} missing ({col_info['missing_percent']:.1f}%)")
    
    # Remover peso_materno (se existir) pois foi removido do Modelo 1
    if 'peso_materno' in df.columns:
        print(f"\n🗑️ Removendo 'peso_materno' (variável do momento da pesquisa)")
        df = df.drop('peso_materno', axis=1)
    
    # Criar dataset apenas com casos completos
    print(f"\n🔍 Criando dataset com casos completos...")
    initial_count = len(df)
    
    # Remover linhas com qualquer valor missing
    df_complete = df.dropna()
    
    final_count = len(df_complete)
    lost_count = initial_count - final_count
    retention_rate = (final_count / initial_count) * 100
    
    print(f"  - Registros iniciais: {initial_count:,}")
    print(f"  - Registros após remoção: {final_count:,}")
    print(f"  - Registros perdidos: {lost_count:,}")
    print(f"  - Taxa de retenção: {retention_rate:.1f}%")
    
    # Verificar se há missing values restantes
    remaining_missing = df_complete.isnull().sum().sum()
    print(f"  - Missing values restantes: {remaining_missing}")
    
    # Analisar distribuição das variáveis target
    target_vars = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']
    print(f"\nDistribuição das categorias target no Modelo 3:")
    for var in target_vars:
        if var in df_complete.columns:
            count = df_complete[var].sum()
            percent = (count / len(df_complete)) * 100
            original_count = df[var].sum() if var in df.columns else 0
            retention = (count / original_count * 100) if original_count > 0 else 0
            print(f"  - {var.capitalize()}: {count} ({percent:.1f}%) - {retention:.1f}% do original")
    
    # Identificar tipos de variáveis
    target_cols = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']
    feature_cols = [col for col in df_complete.columns if col not in target_cols + ['id_anon']]
    
    print(f"\nComposição do Modelo 3:")
    print(f"  - Features: {len(feature_cols)} colunas")
    print(f"  - Target variables: {len(target_cols)} colunas")
    print(f"  - ID: 1 coluna (id_anon)")
    print(f"  - Total: {df_complete.shape[1]} colunas")
    
    # Categorizar features por origem
    model1_features = []  # Demográficas/perinatais
    model2_features = []  # Alimentação/amamentação
    
    # Features que estavam no Modelo 1 (demográficas/perinatais)
    demographic_keywords = ['regiao_', 'cor_', 'parto_', 'sexo_', 'idade_', 'semanas_', 'peso_nascimento', 
                           'altura_nascimento', 'tipo_de_parto', 'chupeta_', 'sabe_ler', 'sindrome_', 
                           'teve_exposicao', 'nivel_educacional', 'prenatal_consultas', 'recebe_beneficio', 
                           'vd_ien_escore']
    
    # Features que estavam no Modelo 2 (alimentação)
    feeding_keywords = ['k08_', 'k07_', 'k06_', 'k11_', 'k03_', 'k04_', 'k15_', 'tempo_primeira', 
                       'usou_utensilios', 'dias_aleitamento', 'doou_leite', 'recebeu_leite', 
                       'amamentou_outra', 'usou_mamadeira', 'deixou_amamentar', 'busca_info', 
                       'deu_outros_liquidos']
    
    for feature in feature_cols:
        is_demographic = any(keyword in feature for keyword in demographic_keywords)
        is_feeding = any(keyword in feature for keyword in feeding_keywords)
        
        if is_demographic:
            model1_features.append(feature)
        elif is_feeding:
            model2_features.append(feature)
        else:
            # Classificar manualmente features ambíguas
            if feature in ['k05_prenatal_consultas']:  # Esta estava no Modelo 1
                model1_features.append(feature)
            else:
                model2_features.append(feature)
    
    print(f"\nClassificação das features:")
    print(f"  - Origem Modelo 1 (demográficas/perinatais): {len(model1_features)} features")
    print(f"  - Origem Modelo 2 (alimentação/amamentação): {len(model2_features)} features")
    
    # Corrigir tipos de dados das variáveis binárias (principalmente do Modelo 2)
    binary_vars = []
    for col in df_complete.columns:
        if col in feature_cols:
            unique_vals = df_complete[col].dropna().unique()
            if len(unique_vals) == 2 and set(unique_vals).issubset({0, 1, 0.0, 1.0}):
                binary_vars.append(col)
                df_complete[col] = df_complete[col].astype(int)
    
    print(f"\nCorreção de tipos de dados:")
    print(f"  - Variáveis binárias identificadas e convertidas: {len(binary_vars)}")
    
    # Salvar o dataset do Modelo 3
    output_dir = '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split'
    os.makedirs(output_dir, exist_ok=True)
    
    output_path = os.path.join(output_dir, 'DSModel3.csv')
    df_complete.to_csv(output_path, index=False)
    
    print(f"\n✅ DATASET MODELO 3 CRIADO:")
    print(f"  - Caminho: {output_path}")
    print(f"  - Dimensões: {df_complete.shape}")
    print(f"  - Features: {len(feature_cols)}")
    print(f"  - Casos completos: {len(df_complete):,}")
    
    # Comparação com modelos anteriores
    print(f"\n" + "="*60)
    print("COMPARAÇÃO ENTRE MODELOS")
    print("="*60)
    
    print(f"MODELO 1 (Demográficas/Perinatais):")
    print(f"  - Amostra: 8,236 crianças (100%)")
    print(f"  - Features: ~27-28")
    print(f"  - Missing: 0%")
    
    print(f"\nMODELO 2 (Práticas Alimentares):")
    print(f"  - Amostra: ~5,638 crianças (68.5%)")
    print(f"  - Features: ~17-18")
    print(f"  - Missing: 0% (após exclusão)")
    
    print(f"\nMODELO 3 (Combinado):")
    print(f"  - Amostra: {len(df_complete):,} crianças ({retention_rate:.1f}%)")
    print(f"  - Features: {len(feature_cols)}")
    print(f"  - Missing: 0% (casos completos)")
    print(f"  - Combina: {len(model1_features)} demográficas + {len(model2_features)} alimentares")
    
    # Calcular obesidade para comparação
    if 'obeso' in df_complete.columns:
        obese_count = df_complete['obeso'].sum()
        obese_percent = (obese_count / len(df_complete)) * 100
        print(f"  - Obesos: {obese_count} ({obese_percent:.1f}%)")
    
    print(f"\n💡 PRÓXIMOS PASSOS:")
    print(f"  1. Train-test split estratificado para Modelo 3")
    print(f"  2. Regularização (Lasso/Ridge/Elastic Net)")
    print(f"  3. Seleção de algoritmos ML")
    print(f"  4. Comparação com Modelos 1 e 2")
    
    return df_complete, {
        'model1_features': model1_features,
        'model2_features': model2_features,
        'binary_vars': binary_vars,
        'retention_rate': retention_rate,
        'feature_count': len(feature_cols)
    }

# Executar criação do Modelo 3
if __name__ == "__main__":
    model3_data, model3_info = create_model3_combined_dataset()

CRIAÇÃO DO MODELO 3 - DATASET COMBINADO
Dataset original:
  - Dimensões: (8236, 50)
  - Total de colunas: 50
  - Total de registros: 8,236

Análise de missing values:
  - Colunas SEM missing: 33
  - Colunas COM missing: 17

Colunas com missing values:
  - k03_prenatal: 2121 missing (25.8%)
  - k04_prenatal_semanas: 2226 missing (27.0%)
  - k06_peso_engravidar: 2108 missing (25.6%)
  - k07_peso_final: 2106 missing (25.6%)
  - k08_quilos: 2104 missing (25.5%)
  - k11_amamentou: 2113 missing (25.7%)
  - tempo_primeira_mamada_horas: 2379 missing (28.9%)
  - k15_recebeu: 2441 missing (29.6%)
  - deu_outros_liquidos: 2440 missing (29.6%)
  - dias_aleitamento_exclusivo: 2379 missing (28.9%)
  - doou_leite_banco: 2379 missing (28.9%)
  - recebeu_leite_banco: 2379 missing (28.9%)
  - amamentou_outra_crianca: 2379 missing (28.9%)
  - deixou_amamentar_por_outra: 2393 missing (29.1%)
  - usou_utensilios_amamentacao: 2379 missing (28.9%)
  - usou_mamadeira: 2389 missing (29.0%)
  - busca_info_aleit

/var/folders/5y/0szcmfss733cjc3_cyqw8lj80000gn/T/ipykernel_19371/475148683.py:138: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_complete[col] = df_complete[col].astype(int)


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

def create_train_test_split_model3():
    """
    Cria train-test split estratificado para o Modelo 3 (dataset combinado)
    mantendo a mesma proporção das categorias target
    """
    
    # Carregar o dataset do Modelo 3
    model3_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/DSModel3.csv'
    df = pd.read_csv(model3_path)
    
    print("="*80)
    print("TRAIN-TEST SPLIT ESTRATIFICADO - MODELO 3")
    print("="*80)
    
    # Informações do dataset
    print(f"Dataset Modelo 3:")
    print(f"  - Dimensões: {df.shape}")
    print(f"  - Total de registros: {len(df):,}")
    
    # Definir variáveis target
    target_vars = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']
    
    # Verificar se todas as variáveis target existem
    missing_targets = [var for var in target_vars if var not in df.columns]
    if missing_targets:
        print(f"Erro: Variáveis target não encontradas: {missing_targets}")
        return None, None
    
    # Criar variável target única para stratificação
    # Converter matriz binária em categoria única (0=desnutrido, 1=eutrofico, 2=sobrepeso, 3=obeso)
    target_combined = (df['desnutrido'] * 0 + 
                      df['eutrofico'] * 1 + 
                      df['sobrepeso'] * 2 + 
                      df['obeso'] * 3)
    
    # Verificar distribuição original das classes
    print(f"\nDistribuição original das classes:")
    for i, var in enumerate(target_vars):
        count = df[var].sum()
        percent = (count / len(df)) * 100
        print(f"  - {var.capitalize()}: {count:,} ({percent:.1f}%)")
    
    print(f"  - Total: {target_combined.value_counts().sum():,}")
    
    # Preparar features (todas exceto target variables)
    feature_cols = [col for col in df.columns if col not in target_vars]
    X = df[feature_cols].copy()
    y = target_combined.copy()
    ids = df['id_anon'].copy()
    
    print(f"\nFeatures selecionadas: {len(feature_cols)} colunas")
    print(f"  - Inclui id_anon: {'id_anon' in feature_cols}")
    print(f"  - Features totais: {len(feature_cols) - 1}")  # -1 por causa do id_anon
    
    # Implementar train-test split estratificado
    print(f"\nExecutando train-test split estratificado...")
    print(f"  - Proporção: 80% train / 20% test")
    print(f"  - Random state: 42")
    print(f"  - Stratify: pelas 4 classes nutricionais")
    
    X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
        X, y, ids,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    
    # Reconverter y_train e y_test de volta para matriz binária
    def convert_to_binary_matrix(y_series, target_vars):
        """Converte target único de volta para matriz binária"""
        n_samples = len(y_series)
        matrix = pd.DataFrame(0, index=y_series.index, columns=target_vars)
        
        for i, var in enumerate(target_vars):
            matrix.loc[y_series == i, var] = 1
        
        return matrix
    
    y_train_matrix = convert_to_binary_matrix(y_train, target_vars)
    y_test_matrix = convert_to_binary_matrix(y_test, target_vars)
    
    # Criar datasets finais combinando features + target
    train_final = pd.concat([X_train, y_train_matrix], axis=1)
    test_final = pd.concat([X_test, y_test_matrix], axis=1)
    
    print(f"\nResultados do split:")
    print(f"Training set: {len(train_final):,} observações ({len(train_final)/len(df)*100:.1f}%)")
    print(f"Test set: {len(test_final):,} observações ({len(test_final)/len(df)*100:.1f}%)")
    
    # Verificar distribuição das classes em cada conjunto
    print(f"\nDistribuição no TRAINING SET:")
    for var in target_vars:
        count = train_final[var].sum()
        percent = (count / len(train_final)) * 100
        print(f"  - {var.capitalize()}: {count:,} ({percent:.1f}%)")
    
    print(f"\nDistribuição no TEST SET:")
    for var in target_vars:
        count = test_final[var].sum()
        percent = (count / len(test_final)) * 100
        print(f"  - {var.capitalize()}: {count:,} ({percent:.1f}%)")
    
    # Quality Assurance
    print(f"\nQuality Assurance:")
    
    # Verificar sobreposição de IDs
    id_overlap = set(ids_train) & set(ids_test)
    print(f"  - ID overlap entre train/test: {len(id_overlap)} (deve ser 0)")
    
    # Verificar preservação total da amostra
    total_samples = len(train_final) + len(test_final)
    print(f"  - Total de amostras preservadas: {total_samples:,} de {len(df):,}")
    print(f"  - Preservação: {'100%' if total_samples == len(df) else 'Erro'}")
    
    # Verificar se cada linha tem exatamente 1 categoria ativa
    train_sums = train_final[target_vars].sum(axis=1)
    test_sums = test_final[target_vars].sum(axis=1)
    
    print(f"  - Linhas train com exatamente 1 categoria: {(train_sums == 1).sum()} de {len(train_final)}")
    print(f"  - Linhas test com exatamente 1 categoria: {(test_sums == 1).sum()} de {len(test_final)}")
    
    # Comparar proporções originais vs. splits
    print(f"\nComparação de proporções:")
    print(f"{'Categoria':<12} {'Original':<10} {'Train':<10} {'Test':<10} {'Status':<8}")
    print("-" * 55)
    
    for var in target_vars:
        orig_pct = (df[var].sum() / len(df)) * 100
        train_pct = (train_final[var].sum() / len(train_final)) * 100
        test_pct = (test_final[var].sum() / len(test_final)) * 100
        
        # Verificar se as proporções são similares (±0.5%)
        status = "✓" if abs(orig_pct - train_pct) < 0.5 and abs(orig_pct - test_pct) < 0.5 else "⚠"
        
        print(f"{var.capitalize():<12} {orig_pct:<10.1f} {train_pct:<10.1f} {test_pct:<10.1f} {status:<8}")
    
    # Salvar os datasets
    output_dir = '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split'
    
    train_path = os.path.join(output_dir, 'MODEL3TRAIN.csv')
    test_path = os.path.join(output_dir, 'MODEL3TEST.csv')
    
    train_final.to_csv(train_path, index=False)
    test_final.to_csv(test_path, index=False)
    
    print(f"\nDatasets salvos:")
    print(f"  - Training: {train_path}")
    print(f"    Dimensões: {train_final.shape}")
    print(f"  - Test: {test_path}")
    print(f"    Dimensões: {test_final.shape}")
    
    # Comparação com modelos anteriores
    print(f"\n" + "="*80)
    print("COMPARAÇÃO COM MODELOS ANTERIORES")
    print("="*80)
    
    print(f"MODELO 1 (Demográficas/Perinatais):")
    print(f"  - Train: 6,588 registros × 32 colunas")
    print(f"  - Test: 1,648 registros × 32 colunas")
    print(f"  - Features: 28")
    
    print(f"\nMODELO 2 (Práticas Alimentares):")
    print(f"  - Train: 4,510 registros × 22 colunas")
    print(f"  - Test: 1,128 registros × 22 colunas") 
    print(f"  - Features: 18")
    
    print(f"\nMODELO 3 (Combinado):")
    print(f"  - Train: {len(train_final):,} registros × {train_final.shape[1]} colunas")
    print(f"  - Test: {len(test_final):,} registros × {test_final.shape[1]} colunas")
    print(f"  - Features: {len(feature_cols) - 1}")  # -1 por causa do id_anon
    print(f"  - Combinação: Todas as variáveis disponíveis")
    
    # Expectativas para análise
    print(f"\n" + "="*60)
    print("EXPECTATIVAS PARA ANÁLISE")
    print("="*60)
    print(f"Hipóteses a testar:")
    print(f"  - Modelo 3 > Modelo 2 (0.598 AUC-ROC): Mais informação disponível")
    print(f"  - Modelo 3 > Modelo 1 (0.570 AUC-ROC): Inclui variáveis alimentares")
    print(f"  - Limitação: Mesmo com 44 features, pode não superar AUC ~0.6")
    print(f"  - Risco: Overfitting com mais variáveis")
    
    print(f"\nPróximos passos:")
    print(f"  1. Regularização (Lasso/Ridge/Elastic Net)")
    print(f"  2. Seleção de algoritmos ML")
    print(f"  3. Comparação de performance")
    print(f"  4. Análise de features importantes")
    
    return train_final, test_final

# Executar o train-test split
if __name__ == "__main__":
    train_data, test_data = create_train_test_split_model3()

TRAIN-TEST SPLIT ESTRATIFICADO - MODELO 3
Dataset Modelo 3:
  - Dimensões: (5638, 49)
  - Total de registros: 5,638

Distribuição original das classes:
  - Desnutrido: 157 (2.8%)
  - Eutrofico: 4,995 (88.6%)
  - Sobrepeso: 318 (5.6%)
  - Obeso: 168 (3.0%)
  - Total: 5,638

Features selecionadas: 45 colunas
  - Inclui id_anon: True
  - Features totais: 44

Executando train-test split estratificado...
  - Proporção: 80% train / 20% test
  - Random state: 42
  - Stratify: pelas 4 classes nutricionais

Resultados do split:
Training set: 4,510 observações (80.0%)
Test set: 1,128 observações (20.0%)

Distribuição no TRAINING SET:
  - Desnutrido: 126 (2.8%)
  - Eutrofico: 3,996 (88.6%)
  - Sobrepeso: 254 (5.6%)
  - Obeso: 134 (3.0%)

Distribuição no TEST SET:
  - Desnutrido: 31 (2.7%)
  - Eutrofico: 999 (88.6%)
  - Sobrepeso: 64 (5.7%)
  - Obeso: 34 (3.0%)

Quality Assurance:
  - ID overlap entre train/test: 0 (deve ser 0)
  - Total de amostras preservadas: 5,638 de 5,638
  - Preservação: 100